# Data Description
- Author: 詹明翰、謝子詮、曾翊涵；共同作者不等於已完成個別工作。
- Date first created: 20261127
- Background: Unit2.6唯一固定456里Dataset B3，供事故分群後交通環境比較，不直接入Dataset A主要分群。七候選及必要敏感度、分子分母、來源時間／品質、零／NA／Blocked與逐特徵readiness均保留。D原版為主，D改良版／Prediction暫緩。
- Unit of observation: 一個固定研究行政里；village_id為10碼CPTVID唯一鍵，456列／12行政區，沒有annual／pooled／S0–S3重複。
- Sample period: 研究事故期間2024–2025，B3來源則為混合固定快照：OSM README 2025-01-01T21:21:00Z；號誌檔名20230706、資源2025-07-15、實際觀測日未認證；行穿2024-07-12封裝與2026資源頁更新分開；路寬2023-12-28資源／XML2021；MRT2026-08-04。不是同時點完整普查、兩套年度設施觀測或事故前環境。
- Known issues: 固定原里界孔洞／微量重疊不修補，所有面積分母沿Unit2.2。各來源原CRS獨立核對：OSM PRJ4326、行穿／路寬PRJ3826；號誌／MRT分別採有既有證據的4326工作解讀，非一律指定3826或正式datum認證。道路是指定OSM線表示，原major六類為研究狹義分組，非臺灣法定等級；union可能合併共位置立體表示，保留平行車道；共里界線段從各里主線長扣除，inclusive／shared及市域和分開。號誌主要A為可讀非機房測試、唯一內部配對且同座標有效成員皆來源區名已識別吻合；B不要求區名一致，未知與明確衝突分開，LCS不直接判非交通設施，A／B另存排除LCS-only敏感度。精確座標去重不是實體路口去重，不用近距離合併或最近里。行穿只算六層面union覆蓋，不當完整設施數、穿越長度或安全完整性。路寬是計畫／地籍尺寸標註樣本，精確正規化geometry＋width去重，採中點唯一歸里、跨里整線另稽核；不是實測通行寬度、縱向長度代表樣本或車道數，不以尺寸線長加權。MRT是出入口不是站數或人次；全部有效市外點參與300／500／800m先union buffer，quad_segs16；500m操作候選非認證通用步行門檻，不是步行路網／通勤暴露，2026資料不進事前預測。混合標線仍Blocked，未重開人行道分類或補行穿；所有Blocked數值欄未建立。觀測0不證明現實無設施，無路寬樣本為NA，處理失敗另flag，不補值或完整案例刪里。面積QA0.1m²、線長0.1m、覆蓋比例以0.1m²/A、major比例1e-10容忍，原比例及浮點超界保留，不clip或刪極端值。B3不是交通暴露、VKT或真正事故風險，不支持因果解釋。本文件只獨立讀saved files與必要鍵，不重跑GIS／舊Notebook。成員交付確認與DA／PM審查待完成。
- Definition for each variable: 下方完整113欄字典列dtype／定義／單位／來源／缺失規則；七核心與敏感度Readiness Matrix另列觀察單位、時點、CRS、分子／分母、完整性及用途。
- Data path: `data/processed/taipei_dataset_b3.csv`
- Source data: 精確29項來源／helper及14項僅交付鍵來源由B3_INPUTS／HANDOFF_ONLY_INPUTS逐檔固定SHA列明；六原始來源、固定界五配套、正式里主表及schema／說明、Unit1.7 config／helper／主驗證與六raw說明／封存證據、既有里驗證與處理；Unit2.5 helper僅reader／hash／顯示utility，不使用B2特徵值。
- Processing code: [處理Notebook](../code/prepare_taipei_dataset_b3.ipynb)；[helper](../code/unit26_transportation_support.py)。Producer SHA `c10ccbd944f96710af4cbb25b7bb980bf80ef50e109ecbae8a40a863880c4f06`。
- Role: external_environment_transportation；事故分群後外部比較，是否正式分析使用仍待DA及學生審查。
- Verification status: 正式456×113，717,179 bytes；處理／分母／來源／空間品質及精確讀回已通過；本文件獨立乾淨kernel結果見Assertions。來源完整或人工驗收不由技術通過推定。
- SHA-256: `36d6f42a86a54ba39552c611332c5185c89a3740e14f5e9e0268158ec8d9debe`
- Schema: `data/processed/taipei_dataset_b3_schema.json`；SHA `440f6020e336990071a3492eba2f42ea36229f11948a8ea14f59427c9e25b8bf`。含道路分類metadata、來源原CRS／時間／品質、逐特徵Readiness與Blocked原因。
- Downstream analysis input: `data/processed/taipei_dataset_b3.csv`（與Data path同檔），無第二份B3分析資料。

Primary Owner：謝子詮（DE）；Collaborator / Analytical Reviewer：詹明翰（DA）；Project Reviewer：曾翊涵（PM）。Deliverables：B3 CSV／schema／本說明、主處理及必要helper。Actual Contribution：Codex協助，三位共同作者保留，成員實際交付／驗收／審查待據實補錄。

資料授權：© OpenStreetMap contributors，ODbL1.0，Geofabrik extract；其他政府來源依原說明。Unit1.7封存的Geofabrik格式PDF為存取失敗紀錄，原包不改；本次有限查閱[官方通用格式說明](https://download.geofabrik.de/osm-data-in-gis-formats-free.pdf)2026-04-02版，只輔助欄位與標籤語義，不認證2025匯出全部語義同版，也不改Unit1.7研究major六類。


## Packages、精確來源與正式版本

In [1]:
from pathlib import Path
import sys,json,hashlib,platform
import pandas as pd
import numpy as np
from IPython.display import display,Markdown
for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents):
    if (p/'AGENTS.md').is_file() and (p/'templates').is_dir():
        PROJECT_ROOT=p;break
else:raise FileNotFoundError('請從研究工作包內執行')
assert Path(sys.executable).resolve()==(PROJECT_ROOT/'.venv/Scripts/python.exe').resolve()
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as stream:
        for part in iter(lambda:stream.read(4*1024*1024),b''):h.update(part)
    return h.hexdigest()
PROJECT_CONTEXT={'path': 'PROJECT.md', 'before_unit26_sha256': '9edafab5f10a3cd6a4b4157eecf6f65f1a95ecd6c9ce71c1dd6bf18f566ec91f', 'role': 'reviewed mutable research context, authorized updates in Unit2.6; not transportation numeric input'}
B3_INPUTS={'data/raw/taiwan-250101-free.shp.zip': 'ad19e79129efc0a447db5fe5d684e732199a900251f04f2aacef821cfa9ff1ab', 'data/raw/號誌路口經緯度一覽20230706.csv': '4de42a369f6b6ba38df29ff2d357f8ad5aec7a237b8709edc431d77c3cfb29a7', 'data/raw/(交工處)行人穿越線圖資_202407121519.zip': 'a7f4241ea1ecccfb09050fb38f4d8537a7f5ec1d4134eb7b8929217601ba0493', 'data/raw/路寬檔_提供.zip': 'a92d7b7225509489181606158d1d9cbe3b1a4c83b120ee12d577460109852d70', 'data/raw/(交工處)標線型人行道圖資_202610300914.zip': 'c17a853905ee242f88a9cbdefd4cf59130c08ab0eae40495365b6d36857a70a9', 'data/raw/臺北捷運車站出入口座標_1150804.csv': '134d8488c240fb23e04535db6a94da73006f890c83617eeedaa9b26c9f120015', 'data/raw/G97_A_CAVLGE_P.shp': 'f7fb4813a2283198ca564f5686823c454c5fd832d2f6981a2ee88a94719be3cc', 'data/raw/G97_A_CAVLGE_P.shx': 'ab7bc5e63229f676d3580b68e98db59b91a2d327c74ca23b0000ca1c568d414e', 'data/raw/G97_A_CAVLGE_P.dbf': '2f43f92e84279a45b2512cf453e9b72f706d1ad1402665c946b1a354f75d8577', 'data/raw/G97_A_CAVLGE_P.prj': '8dac7d82ea889063f9eaa7df408fcaeb7307d32831ea973b6534d0139a4523c5', 'data/raw/G97_A_CAVLGE_P.cpg': '3ad3031f5503a4404af825262ee8232cc04d4ea6683d42c5dd0a2f2a27ac9824', 'data/processed/taipei_village_base.csv': 'db0a225df3568ddb2a47eab1e1d28843e1594dec37956c57eea926e444968209', 'data/processed/taipei_village_base_schema.json': 'a7e5e3b4d77ab506736ecc940e26a87df6cace3a738cbdaeef1e835e6bed7070', 'data/processed/taipei_village_base_documentation.ipynb': 'a843d587788b4ea6a943cc05d9413908cf8b66b2f73f9fdbec01e2ab52edd535', 'data/code/taipei_transportation_infrastructure_validation.ipynb': '5fa4a342b2f49359c560929ce9aecf40f6f09e285dc05447b56af94c448e8508', 'data/code/unit17_validation_config.json': '7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7', 'data/code/unit17_validation_support.py': '1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080', 'data/raw/unit17_osm_raw_documentation.ipynb': 'e71a3c28d0edf1307e0842d61b9371df34fd1b46be5840532ce7784ba7e37294', 'data/raw/unit17_signals_raw_documentation.ipynb': '3f38193c3dde1a0ad4034889a93fd7df73c60d890bc2a2e9a3411e8caa5b24a4', 'data/raw/unit17_crossings_raw_documentation.ipynb': 'b930a5a38df43b426c1982359d34e9676e59d57eccfa76c029d7a6819a8e1289', 'data/raw/unit17_road_width_raw_documentation.ipynb': 'b34d11b2b57ba3f3e974fde75cfdaeec831089e9737cba18fd0a66af383ce242', 'data/raw/unit17_sidewalk_raw_documentation.ipynb': '79ed45e4d846d75709895565d396ac802676489ea6cc89caa238117634697a48', 'data/raw/unit17_mrt_raw_documentation.ipynb': '215cb596f1c0b8c3aefedc5bda3d7b53ce6230b4d753db4d22d90e3e3a613d85', 'data/raw/unit17_supplementary_source_evidence.zip': 'f5394c215f0db5a0124ea831d0552fe9aa26ea1c5901797a5a8fcce042f6b3bf', 'data/raw/unit17_supplementary_source_evidence_documentation.ipynb': '92726536d73af85ea8ad9dd884ed64ad92d9c8f67c203fcd4a7097be7ed6deba', 'data/code/taipei_village_boundary_validation.ipynb': '662dbee2dc42c3552b12628a1e89fcc81b07155c225277117f50d07e0145a343', 'data/code/prepare_taipei_village_base_and_crash_spatial_assignment.ipynb': 'ef8a3d21f38ecb902f9495720b0094363ebd57cf444b837176fe2b815992abc4', 'data/code/unit25_zoning_support.py': 'a5d1a8bbc8b0473921340b01af09506902123030502a61035def77c83af7b2a4', 'data/code/unit26_transportation_support.py': 'dcdbeb261cec78d6eb3d967755da79607d25186671dea3988d61696db7f6c557'}
HANDOFF_ONLY_INPUTS={'data/processed/taipei_dataset_a_annual.csv': '4e264176c970263777eb4c564f427e1f839a45b31552a37bc4bdbfbed6b09422', 'data/processed/taipei_dataset_a_annual_schema.json': '4d0c5b7b307b1cca4261a9595cd1ca894f549d88e06bed1b0b071dcf7c40fb21', 'data/processed/taipei_dataset_a_pooled.csv': '081c7dd42c94e16a5684f27edaebd310a9c556bae7460783dd87fde8d26ea46a', 'data/processed/taipei_dataset_a_pooled_schema.json': '7878146fac39d4bfabee965b6001d0cf5e101ca1d76778599fec4b480ad59b5a', 'data/processed/taipei_dataset_b1_annual.csv': 'd4ad76e82ad6f00d5b146c00c0e065d2095cd9fd36fe6bbbad5feb40705ae795', 'data/processed/taipei_dataset_b1_annual_schema.json': '89bb5ace9a7abf9bcedcc1ff0d6b2f91d45578f7c6cfe20a685bd6802d8fa7ca', 'data/processed/taipei_dataset_b1_pooled.csv': '3b02fb7b338b59f2552972ccb98efa0d723e525826b301cdda021efa9ddd4e50', 'data/processed/taipei_dataset_b1_pooled_schema.json': '63f46e1ae0cba9c19549ee5f8b9ad39084a22f5d2d738b62567b1b3f5e86f340', 'data/processed/taipei_dataset_c_annual.csv': '9ab1970ed0f790ebdf31e1726094493305da00ac4efaf84b8a554f7debfa560e', 'data/processed/taipei_dataset_c_annual_schema.json': '4a8151a9a5dcd716cc03312b303c4455cb47210192c43b86cf3a0e06d268cb6f', 'data/processed/taipei_dataset_c_pooled.csv': 'caceea61ff906b3e2f0b4796a31af3373a929307a0c7f7094fb9373794f76bc6', 'data/processed/taipei_dataset_c_pooled_schema.json': '44baac19975650d792e23e81c5463b84e00f299c9c606ef0f95abf8b3756d3ab', 'data/processed/taipei_dataset_b2.csv': '1af7ce133e62a1188e8755a1e4e7bc55db8b89d639f089f1c75319c8a9ee0269', 'data/processed/taipei_dataset_b2_schema.json': 'd5857362444df6bea1f606330d0363d792cc1a7f56812c9bb4917ce1b5e3a102'}
for rel,h in {**B3_INPUTS,**HANDOFF_ONLY_INPUTS}.items():assert sha(PROJECT_ROOT/rel)==h,('INPUT HASH MISMATCH',rel)
sys.path.insert(0,str(PROJECT_ROOT/'data/code'))
import unit26_transportation_support as u
print('Python',platform.python_version(),'來源／helper',len(B3_INPUTS),'僅交付鍵來源',len(HANDOFF_ONLY_INPUTS))

EXPECTED_OUTPUTS={'data/processed/taipei_dataset_b3.csv': '36d6f42a86a54ba39552c611332c5185c89a3740e14f5e9e0268158ec8d9debe', 'data/processed/taipei_dataset_b3_schema.json': '440f6020e336990071a3492eba2f42ea36229f11948a8ea14f59427c9e25b8bf'}
assert sha(PROJECT_ROOT/'data/code/prepare_taipei_dataset_b3.ipynb')=='c10ccbd944f96710af4cbb25b7bb980bf80ef50e109ecbae8a40a863880c4f06'
for rel,h in EXPECTED_OUTPUTS.items():assert sha(PROJECT_ROOT/rel)==h,('OUTPUT HASH MISMATCH',rel)
print('正式CSV／schema及最終producer／helper版本一致；只讀saved files')


Python 3.12.2 來源／helper 29 僅交付鍵來源 14
正式CSV／schema及最終producer／helper版本一致；只讀saved files


## Change for Your Own Data

先核對SHA，再按schema讀nullable dtype與string ID；`__MISSING__`與原空字串分開，不猜版本或以glob選檔。

In [2]:
contract=json.loads((PROJECT_ROOT/'data/processed/taipei_dataset_b3_schema.json').read_text(encoding='utf-8'))
assert contract['source_inputs']==B3_INPUTS and contract['handoff_only_inputs']==HANDOFF_ONLY_INPUTS
frame=u.read_csv(PROJECT_ROOT/'data/processed/taipei_dataset_b3.csv',contract)
base_contract=json.loads((PROJECT_ROOT/'data/processed/taipei_village_base_schema.json').read_text(encoding='utf-8'))
base=u.read_csv(PROJECT_ROOT/'data/processed/taipei_village_base.csv',base_contract)
u.validate_dataset(frame,base)
assert frame.shape==(456, 113)
print('Rows',len(frame),'Columns',frame.shape[1],'primary key',contract['primary_key'])
u.show('來源原CRS及工作CRS，點層B級假設另列',pd.DataFrame([{'source':k,**v} for k,v in contract['source_crs'].items()]))
u.show('來源時間與官方欄位語義，非兩套年度',pd.DataFrame([{'source':k,**v} for k,v in contract['source_metadata'].items()]))


Rows 456 Columns 113 primary key ['village_id']


**來源原CRS及工作CRS，點層B級假設另列**

,source,raw,work,evidence_grade,evidence
0,osm,EPSG:4326,EPSG:3826,A metadata,actual archive PRJ; coordinate accuracy itself not certified
1,crossings,EPSG:3826,EPSG:3826,A metadata,actual archive PRJ; coordinate accuracy itself not certified
2,road_width,EPSG:3826,EPSG:3826,A metadata,actual archive PRJ; coordinate accuracy itself not certified
3,signals,EPSG:4326 working interpretation,EPSG:3826,B working convention,Unit1.7 source-specific coordinate fields and official page; no file-specific datum certification
4,mrt,EPSG:4326 working interpretation,EPSG:3826,B working convention,Unit1.7 source-specific coordinate fields and official page; no file-specific datum certification


**來源時間與官方欄位語義，非兩套年度**

,source,title,provider,url,publication,period,limitation,license
0,osm,Geofabrik OSM道路,Geofabrik／OpenStreetMap contributors,https://download.geofabrik.de/asia/taiwan.html,歷史資源檔名250101；精確發布時間未認證,包內README明示2025-01-01T21:21:00Z快照；非2024／2025逐年獨立路網,僅讀道路及研究區外圍bbox；線表示密度非實體中心線、交通流量或VKT；免費格式欠完整OSM節點拓樸,Open Database License 1.0；需OpenStreetMap contributors標示及遵守ODbL
1,signals,號誌路口位置,臺北市交通局交通管制工程處,https://data.taipei/dataset/detail?id=4a599738-6550-448a-9a78-03b26c67e249,資源更新2025-07-15,檔名20230706非每筆觀測／設置時間；實際快照時點仍未認證,原檔為tab分隔；路口位置紀錄不等於燈頭或設備數，完整歷史號誌存量未證,政府資料開放授權條款第1版；依官方頁，來源與版本須保留
2,crossings,行人穿越線圖資,臺北市交通局交通管制工程處,https://data.taipei/dataset/detail?id=4d6bd0c6-9f56-408b-bb96-ac41134a95d5,目前資源頁更新2026-03-19；本機ZIP成員時間2024-07-12,2024-07-12封裝標記，非全年有效或2025新增完整紀錄；RDDATE需排除1899占位,六個明示行穿線圖層，Polygon構件不先假定完整設施；長寬欄0與單位未證，面積以有效幾何聯集診斷,政府資料開放授權條款第1版；依官方頁，來源與版本須保留
3,road_width,道路路寬標註,臺北市都市發展局,https://data.taipei/dataset/detail?id=068be2c7-eea1-4303-91f8-fbf50c9b2aec,資源2023-12-28；metadata2026-06-17,ZIP內資料修改2023-12-28，XML製作2021；非2024／2025完整年度實測,官方稱依都市計畫參酌地籍標註；線長與標註寬度相近，不能當道路縱向長度作權重或實測車行空間,政府資料開放授權條款第1版；依官方頁，來源與版本須保留
4,sidewalk,標線型人行道來源候選,臺北市交通局交通管制工程處,https://data.taipei/dataset/detail?id=d647b3af-4bbd-4e8f-b420-8f6c1bc1b597,資源2026-10-30,2026-10-30快照；RDDATE僅現存紀錄歷史資格，不含拆除／遷移歷史,實檔含多種LICODE並重現行穿線；缺確切人行道代碼篩選依據，不能全部43016面當人行道,政府資料開放授權條款第1版；依官方頁，來源與版本須保留
5,mrt,臺北捷運出入口,臺北大眾捷運股份有限公司,https://data.taipei/dataset/detail?id=cfa4778c-62c1-497b-b704-756231de348b,資源2026-08-04,1150804檔名及更新快照，無逐出入口設置／撤除日期,固定設施快照，不證明研究年存在；站名與入口分開、保留跨市點，直線緩衝非實際步行可及性,政府資料開放授權條款第1版；依官方頁，來源與版本須保留


## Definition for each variable

In [3]:
dictionary=pd.DataFrame(contract['columns'])
u.show('Definition for each variable：完整字典',dictionary,block=16)
assert list(frame)==dictionary['name'].tolist()
assert [str(frame[c].dtype) for c in frame]==dictionary['dtype'].tolist()

**Definition for each variable：完整字典**

,name,dtype,definition,unit,source,missing_rule
0,village_id,string,Unit2.2固定里主表欄位；verification_status是里碼配對證據，糖廍正常納入,not applicable,formal village base,nonmissing; no eligibility restriction from provisional_match
1,boundary_code,string,Unit2.2固定里主表欄位；verification_status是里碼配對證據，糖廍正常納入,not applicable,formal village base,nonmissing; no eligibility restriction from provisional_match
2,district_name,string,Unit2.2固定里主表欄位；verification_status是里碼配對證據，糖廍正常納入,not applicable,formal village base,nonmissing; no eligibility restriction from provisional_match
3,standard_village_name,string,Unit2.2固定里主表欄位；verification_status是里碼配對證據，糖廍正常納入,not applicable,formal village base,nonmissing; no eligibility restriction from provisional_match
4,verification_status,string,Unit2.2固定里主表欄位；verification_status是里碼配對證據，糖廍正常納入,not applicable,formal village base,nonmissing; no eligibility restriction from provisional_match
5,geometry_quality_flag,string,Unit2.2固定里主表欄位；verification_status是里碼配對證據，糖廍正常納入,not applicable,formal village base,nonmissing; no eligibility restriction from provisional_match
6,fixed_village_area_m2,Float64,完整原固定里面積；非AREA欄，不修界／填洞,m²,Unit2.2 village_area_m2,"positive, nonmissing"
7,fixed_village_area_km2,Float64,人口與設施密度共用原固定面積分母,km²,Unit2.2 village_area_km2,"positive, nonmissing"
8,analysis_role,string,事故分群後外部環境比較，非A主要分群輸入,not applicable,source metadata / Unit1.7 / Unit2.6 specification,known metadata strings retained; no imputation of actual observation dates
9,working_crs,string,各來源依各自證據轉換，並非全部直接指定,not applicable,source metadata / Unit1.7 / Unit2.6 specification,known metadata strings retained; no imputation of actual observation dates


,name,dtype,definition,unit,source,missing_rule
16,marked_sidewalk_readiness,string,混合標線沒有可靠人行道子集；不建立數值,not applicable,source metadata / Unit1.7 / Unit2.6 specification,known metadata strings retained; no imputation of actual observation dates
17,year_specific_observation,boolean,沒有兩套獨立年度B3；不建立S0–S3重複,not applicable,Unit2.6 explicit processing,source-specific missing/zero rules; processing failure => NA
18,mrt_pre_accident_environment_verified,boolean,2026入口不可冒稱2024／2025事故前環境,not applicable,Unit2.6 explicit processing,source-specific missing/zero rules; processing failure => NA
19,osm_road_length_m,Float64,主要指定OSM機動候選線聯集，扣原共同里界線段,m,frozen OSM source + original village polygon,source-specific missing/zero rules; processing failure => NA
20,osm_major_road_length_m,Float64,六個研究major類同規則聯集線長；major為全部候選子集,m,frozen OSM source + original village polygon,source-specific missing/zero rules; processing failure => NA
21,osm_inclusive_road_length_m,Float64,逐里完整覆蓋線長，含共界，不能互斥加總成城市總量,m,frozen OSM source + original village polygon,source-specific missing/zero rules; processing failure => NA
22,osm_shared_boundary_unallocated_length_m,Float64,從本里主要線長扣除的原共同里界線段,m,frozen OSM source + original village polygon,source-specific missing/zero rules; processing failure => NA
23,osm_inclusive_major_road_length_m,Float64,major含共界線長,m,frozen OSM source + original village polygon,source-specific missing/zero rules; processing failure => NA
24,osm_major_shared_boundary_length_m,Float64,major原共界未歸屬線長,m,frozen OSM source + original village polygon,source-specific missing/zero rules; processing failure => NA
25,osm_major_other_overlap_length_m,Float64,major與其他納入類別重合線長；保留分類重疊診斷,m,frozen OSM source + original village polygon,source-specific missing/zero rules; processing failure => NA


,name,dtype,definition,unit,source,missing_rule
32,osm_major_road_ratio,Float64,六類研究major線聯集長／相同全部候選線聯集長；不是官方等級或流量占比,fraction,same-scope major / all road union,zero road denominator => NA; processing failure => NA
33,signal_raw_unique_assigned_record_count,Int64,基本品質過濾前，幾何唯一歸里的原紀錄,source records,frozen signals TAB source / exact-coordinate grouping,source-specific missing/zero rules; processing failure => NA
34,observed_signal_record_count,Int64,屬主要A口徑位置之可讀、非機房測試原紀錄數，尚未座標去重,source records,frozen signals TAB source / exact-coordinate grouping,source-specific missing/zero rules; processing failure => NA
35,observed_unique_signal_location_count,Int64,主要A口徑唯一原數值座標位置數，不是完整實體路口數,coordinate locations,frozen signals TAB source / exact-coordinate grouping,source-specific missing/zero rules; processing failure => NA
36,signal_geometry_only_unique_location_count,Int64,B口徑：相同基本品質與唯一內部幾何，允許區名不一致,coordinate locations,frozen signals TAB source / exact-coordinate grouping,source-specific missing/zero rules; processing failure => NA
37,signal_main_without_lcs_location_count,Int64,A口徑排除僅由LCS紀錄支持的位置，並非認定LCS不是設施,coordinate locations,frozen signals TAB source / exact-coordinate grouping,source-specific missing/zero rules; processing failure => NA
38,signal_geometry_without_lcs_location_count,Int64,B口徑排除僅由LCS支持的位置,coordinate locations,frozen signals TAB source / exact-coordinate grouping,source-specific missing/zero rules; processing failure => NA
39,signal_explicit_test_record_count,Int64,幾何可歸里的機房測試用原紀錄,source records,frozen signals TAB source / exact-coordinate grouping,source-specific missing/zero rules; processing failure => NA
40,signal_lcs_record_count,Int64,幾何可歸里的LCS原紀錄；其實際設備單位尚未認證,source records,frozen signals TAB source / exact-coordinate grouping,source-specific missing/zero rules; processing failure => NA
41,signal_source_district_conflict_record_count,Int64,可判讀來源區名與空間歸區明確不一致的原紀錄,source records,frozen signals TAB source / exact-coordinate grouping,source-specific missing/zero rules; processing failure => NA


,name,dtype,definition,unit,source,missing_rule
48,signal_main_without_lcs_location_density,Float64,A排除LCS-only位置敏感度；唯一觀測位置／完整里面積，不認證實體路口,locations/km²,signal_main_without_lcs_location_count / fixed_village_area_km2,source-specific missing/zero rules; processing failure => NA
49,signal_geometry_without_lcs_location_density,Float64,B排除LCS-only位置敏感度；唯一觀測位置／完整里面積，不認證實體路口,locations/km²,signal_geometry_without_lcs_location_count / fixed_village_area_km2,source-specific missing/zero rules; processing failure => NA
50,observed_signal_locations_per_osm_road_km,Float64,補充：主要A位置／同口徑OSM線表示km，跨來源時點不同,locations/km,main signal locations / osm_road_length_km,zero road denominator => NA
51,observed_crosswalk_area_m2,Float64,六明示行穿標線層union後與原里面交集，非設施數,m²,frozen six-layer crossing source / polygon union,source-specific missing/zero rules; processing failure => NA
52,observed_crosswalk_area_km2,Float64,已觀測行穿標線面積換算,km²,frozen six-layer crossing source / polygon union,source-specific missing/zero rules; processing failure => NA
53,crosswalk_area_ratio,Float64,標線聯集面積／完整里面積，非完整性／安全評分,fraction,frozen six-layer crossing source / polygon union,source-specific missing/zero rules; processing failure => NA
54,crosswalk_raw_intersection_area_sum_m2,Float64,原構件交集面積和，僅比較重複，不當可靠聯集面積,m²,frozen six-layer crossing source / polygon union,source-specific missing/zero rules; processing failure => NA
55,crosswalk_raw_minus_union_area_m2,Float64,原構件交集和減聯集之診斷差，微量負值保留,m²,frozen six-layer crossing source / polygon union,source-specific missing/zero rules; processing failure => NA
56,crosswalk_positive_intersection_source_count,Int64,原Polygon正面積交集筆數，非完整行穿設施數,source polygon components,Unit2.6 explicit processing,source-specific missing/zero rules; processing failure => NA
57,crosswalk_duplicate_geometry_source_intersection_count,Int64,來源正規化重複面中與此里正面積相交之筆數,source polygon components,Unit2.6 explicit processing,source-specific missing/zero rules; processing failure => NA


,name,dtype,definition,unit,source,missing_rule
64,annotated_road_width_assigned_source_record_count,Int64,本里去重樣本對應原始來源筆數，保留追溯,source annotations,frozen planned/cadastral width dimension lines,"no valid samples => count 0, summary NA; never length-weight or impute"
65,annotated_road_width_cross_village_line_sample_count,Int64,中點歸本里但整線與多里有正長度交集之樣本,annotation samples,frozen planned/cadastral width dimension lines,"no valid samples => count 0, summary NA; never length-weight or impute"
66,annotated_road_width_same_geometry_conflict_sample_count,Int64,相同幾何不同寬度之樣本，未任取第一筆,annotation samples,frozen planned/cadastral width dimension lines,"no valid samples => count 0, summary NA; never length-weight or impute"
67,annotated_road_width_same_midpoint_distinct_width_sample_count,Int64,同中點不同寬度旗標，並非一律錯誤或自動合併,annotation samples,frozen planned/cadastral width dimension lines,"no valid samples => count 0, summary NA; never length-weight or impute"
68,annotated_road_width_line_intersection_sample_count,Int64,診斷：精確去重後全線與此里正長相交樣本數，可跨里重複,annotation samples,frozen planned/cadastral width dimension lines,"no valid samples => count 0, summary NA; never length-weight or impute"
69,annotated_road_width_line_intersection_median,Float64,診斷：精確去重後全線相交樣本中位數，不作主值,m,frozen planned/cadastral width dimension lines,"no valid samples => count 0, summary NA; never length-weight or impute"
70,annotated_road_width_excluding_geometry_conflicts_median,Float64,診斷：排除同幾何異寬樣本後中位數，主值不任意合併,m,frozen planned/cadastral width dimension lines,"no valid samples => count 0, summary NA; never length-weight or impute"
71,annotated_road_width_no_valid_samples,boolean,缺標註樣本，統計值為NA而非0,not applicable,Unit2.6 explicit processing,source-specific missing/zero rules; processing failure => NA
72,observed_mrt_entrance_count,Int64,2026快照中唯一內部歸里的出入口紀錄數，非站數,entrance records,frozen MRT CSV,source-specific missing/zero rules; processing failure => NA
73,mrt_entrance_density,Float64,入口紀錄／完整里面積；零入口不等於無可及性,entrances/km²,observed_mrt_entrance_count / fixed_village_area_km2,source-specific missing/zero rules; processing failure => NA


,name,dtype,definition,unit,source,missing_rule
80,mrt_entrance_buffer_800m_area_m2,Float64,全部有效含市外入口800m buffer先union後與原里面相交；quad_segs=16,m²,frozen MRT all-entrance union + original villages,source-specific missing/zero rules; processing failure => NA
81,mrt_entrance_buffer_800m_ratio,Float64,800m直線覆蓋／完整里面積，非步行路網或流量,fraction,buffer union area / fixed_village_area_m2,source-specific missing/zero rules; processing failure => NA
82,mrt_entrance_buffer_800m_crossborder_extra_area_m2,Float64,保留市外入口相較僅市內入口新增覆蓋面積，不重複加總,m²,all-source buffer union minus within-city-entrance buffer union,source-specific missing/zero rules; processing failure => NA
83,coverage_ratio_roundoff_flag,boolean,行穿／MRT覆蓋比例微量超0–1旗標，保留原比例，不作排除或clip,not applicable,Unit2.6 explicit processing,source-specific missing/zero rules; processing failure => NA
84,coverage_ratio_max_excess_m2,Float64,四項覆蓋面積超完整里面積之最大正超額，只作數值診斷,m²,Unit2.6 explicit processing,source-specific missing/zero rules; processing failure => NA
85,osm_major_ratio_roundoff_flag,boolean,major比例微量超界診斷，原比例不clip,not applicable,Unit2.6 explicit processing,source-specific missing/zero rules; processing failure => NA
86,osm_road_density_readiness_status,string,來源與方法的逐特徵技術readiness；不是人工研究驗收,not applicable,feature readiness matrix,source-specific missing/zero rules; processing failure => NA
87,osm_road_density_value_state,string,區分觀測零、無樣本、零分母與處理失敗；不作完整案例刪里,not applicable,actual feature / denominator / sample count,source-specific missing/zero rules; processing failure => NA
88,osm_major_road_ratio_readiness_status,string,來源與方法的逐特徵技術readiness；不是人工研究驗收,not applicable,feature readiness matrix,source-specific missing/zero rules; processing failure => NA
89,osm_major_road_ratio_value_state,string,區分觀測零、無樣本、零分母與處理失敗；不作完整案例刪里,not applicable,actual feature / denominator / sample count,source-specific missing/zero rules; processing failure => NA


,name,dtype,definition,unit,source,missing_rule
96,mrt_entrance_density_readiness_status,string,來源與方法的逐特徵技術readiness；不是人工研究驗收,not applicable,feature readiness matrix,source-specific missing/zero rules; processing failure => NA
97,mrt_entrance_density_value_state,string,區分觀測零、無樣本、零分母與處理失敗；不作完整案例刪里,not applicable,actual feature / denominator / sample count,source-specific missing/zero rules; processing failure => NA
98,mrt_entrance_buffer_500m_ratio_readiness_status,string,來源與方法的逐特徵技術readiness；不是人工研究驗收,not applicable,feature readiness matrix,source-specific missing/zero rules; processing failure => NA
99,mrt_entrance_buffer_500m_ratio_value_state,string,區分觀測零、無樣本、零分母與處理失敗；不作完整案例刪里,not applicable,actual feature / denominator / sample count,source-specific missing/zero rules; processing failure => NA
100,mrt_entrance_buffer_300m_ratio_readiness_status,string,來源與方法的逐特徵技術readiness；不是人工研究驗收,not applicable,feature readiness matrix,source-specific missing/zero rules; processing failure => NA
101,mrt_entrance_buffer_300m_ratio_value_state,string,區分觀測零、無樣本、零分母與處理失敗；不作完整案例刪里,not applicable,actual feature / denominator / sample count,source-specific missing/zero rules; processing failure => NA
102,mrt_entrance_buffer_800m_ratio_readiness_status,string,來源與方法的逐特徵技術readiness；不是人工研究驗收,not applicable,feature readiness matrix,source-specific missing/zero rules; processing failure => NA
103,mrt_entrance_buffer_800m_ratio_value_state,string,區分觀測零、無樣本、零分母與處理失敗；不作完整案例刪里,not applicable,actual feature / denominator / sample count,source-specific missing/zero rules; processing failure => NA
104,signal_geometry_only_location_density_readiness_status,string,來源與方法的逐特徵技術readiness；不是人工研究驗收,not applicable,feature readiness matrix,source-specific missing/zero rules; processing failure => NA
105,signal_geometry_only_location_density_value_state,string,區分觀測零、無樣本、零分母與處理失敗；不作完整案例刪里,not applicable,actual feature / denominator / sample count,source-specific missing/zero rules; processing failure => NA


,name,dtype,definition,unit,source,missing_rule
112,source_complete_census_certified,boolean,所有快照均不冒充同時點完整設施普查,not applicable,Unit2.6 explicit processing,source-specific missing/zero rules; processing failure => NA


## Summary

### Sample

In [4]:
u.show('實際前10里識別／面積／來源欄',frame.iloc[:10,:15])
u.show('實際前10里七候選及值狀態',frame[['village_id']+u.CORE+[c+'_value_state' for c in u.CORE]].head(10))

**實際前10里識別／面積／來源欄**

,village_id,boundary_code,district_name,standard_village_name,verification_status,geometry_quality_flag,fixed_village_area_m2,fixed_village_area_km2,analysis_role,working_crs,osm_snapshot_time,signal_filename_date,signal_observation_date_status,crosswalk_packaging_date,road_width_resource_date
0,6300100002,6300100002,松山區,莊敬里,verified_exact_match,valid_original,1281723.117747,1.281723,external_environment_transportation,EPSG:3826,2025-01-01T21:21:00Z,2023-07-06,not_certified,2024-07-12,2023-12-28
1,6300100003,6300100003,松山區,東榮里,verified_exact_match,valid_original,259347.14737,0.259347,external_environment_transportation,EPSG:3826,2025-01-01T21:21:00Z,2023-07-06,not_certified,2024-07-12,2023-12-28
2,6300100004,6300100004,松山區,三民里,verified_exact_match,valid_original_overlap,134542.285747,0.134542,external_environment_transportation,EPSG:3826,2025-01-01T21:21:00Z,2023-07-06,not_certified,2024-07-12,2023-12-28
3,6300100005,6300100005,松山區,新益里,verified_exact_match,valid_original,115137.5782,0.115138,external_environment_transportation,EPSG:3826,2025-01-01T21:21:00Z,2023-07-06,not_certified,2024-07-12,2023-12-28
4,6300100006,6300100006,松山區,富錦里,verified_exact_match,valid_original,99809.362025,0.099809,external_environment_transportation,EPSG:3826,2025-01-01T21:21:00Z,2023-07-06,not_certified,2024-07-12,2023-12-28
5,6300100007,6300100007,松山區,新東里,verified_exact_match,valid_original_overlap,351945.654552,0.351946,external_environment_transportation,EPSG:3826,2025-01-01T21:21:00Z,2023-07-06,not_certified,2024-07-12,2023-12-28
6,6300100008,6300100008,松山區,富泰里,verified_exact_match,valid_original_overlap,57081.424959,0.057081,external_environment_transportation,EPSG:3826,2025-01-01T21:21:00Z,2023-07-06,not_certified,2024-07-12,2023-12-28
7,6300100009,6300100009,松山區,介壽里,verified_exact_match,valid_original,141203.777408,0.141204,external_environment_transportation,EPSG:3826,2025-01-01T21:21:00Z,2023-07-06,not_certified,2024-07-12,2023-12-28
8,6300100010,6300100010,松山區,精忠里,verified_exact_match,valid_original,2308512.493916,2.308512,external_environment_transportation,EPSG:3826,2025-01-01T21:21:00Z,2023-07-06,not_certified,2024-07-12,2023-12-28
9,6300100011,6300100011,松山區,東光里,verified_exact_match,valid_original,187401.931968,0.187402,external_environment_transportation,EPSG:3826,2025-01-01T21:21:00Z,2023-07-06,not_certified,2024-07-12,2023-12-28


**實際前10里七候選及值狀態**

,village_id,osm_road_density,osm_major_road_ratio,observed_signal_location_density,crosswalk_area_ratio,median_annotated_road_width,mrt_entrance_buffer_500m_ratio,mrt_entrance_density,osm_road_density_value_state,osm_major_road_ratio_value_state,observed_signal_location_density_value_state,crosswalk_area_ratio_value_state,median_annotated_road_width_value_state,mrt_entrance_buffer_500m_ratio_value_state,mrt_entrance_density_value_state
0,6300100002,20.388905,0.349528,7.801997,0.002105,12.0,0.0,0.0,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_zero_not_confirmed_absent,zero_entrances_not_zero_accessibility
1,6300100003,23.924318,0.0,34.702522,0.007523,12.0,0.117961,0.0,observed_positive_with_source_limits,observed_zero_not_confirmed_absent,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_positive_with_source_limits,zero_entrances_not_zero_accessibility
2,6300100004,32.308639,0.0,22.297822,0.010137,8.0,0.0,0.0,observed_positive_with_source_limits,observed_zero_not_confirmed_absent,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_zero_not_confirmed_absent,zero_entrances_not_zero_accessibility
3,6300100005,38.181448,0.0,52.11157,0.016049,8.0,0.0,0.0,observed_positive_with_source_limits,observed_zero_not_confirmed_absent,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_zero_not_confirmed_absent,zero_entrances_not_zero_accessibility
4,6300100006,30.366497,0.0,60.114601,0.011862,18.0,0.0,0.0,observed_positive_with_source_limits,observed_zero_not_confirmed_absent,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_zero_not_confirmed_absent,zero_entrances_not_zero_accessibility
5,6300100007,25.455104,0.136464,31.254825,0.006097,9.0,0.001057,0.0,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_positive_with_source_limits,zero_entrances_not_zero_accessibility
6,6300100008,38.61995,0.0,35.037668,0.012654,8.0,0.0,0.0,observed_positive_with_source_limits,observed_zero_not_confirmed_absent,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_zero_not_confirmed_absent,zero_entrances_not_zero_accessibility
7,6300100009,23.449863,0.0,35.409818,0.009401,12.0,0.0,0.0,observed_positive_with_source_limits,observed_zero_not_confirmed_absent,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_zero_not_confirmed_absent,zero_entrances_not_zero_accessibility
8,6300100010,11.80015,0.0,6.93087,0.00182,12.0,0.297818,1.299538,observed_positive_with_source_limits,observed_zero_not_confirmed_absent,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_positive_with_source_limits
9,6300100011,24.226725,0.0,32.016746,0.006908,8.0,0.903193,5.336124,observed_positive_with_source_limits,observed_zero_not_confirmed_absent,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_positive_with_source_limits,observed_positive_with_source_limits


### Summary Stats

In [5]:
features=[row['feature'] for row in contract['feature_readiness']]
u.show('七核心及MRT／號誌敏感度完整統計',pd.DataFrame(u.feature_statistics(frame,features)))
u.show('分母與標註樣本數分布',pd.DataFrame(u.feature_statistics(frame,['fixed_village_area_km2','osm_road_length_km','observed_signal_record_count','observed_unique_signal_location_count','annotated_road_width_sample_count','observed_mrt_entrance_count'])))
quality=[]
for district,ids in frame.groupby('district_name').groups.items():
    f=frame.loc[ids];a=float(f.fixed_village_area_km2.sum());length=float(f.osm_road_length_km.sum())
    quality.append({'district':district,'villages':len(f),'road_km_per_km2':length/a,'major_ratio_length_weighted':float(f.osm_major_road_length_km.sum()/length) if length>0 else None,'signal_A_locations':int(f.observed_unique_signal_location_count.sum()),'signal_B_locations':int(f.signal_geometry_only_unique_location_count.sum()),'district_conflict_records':int(f.signal_source_district_conflict_record_count.sum()),'width_samples':int(f.annotated_road_width_sample_count.sum()),'width_NA_villages':int(f.median_annotated_road_width.isna().sum()),'mrt_entrances':int(f.observed_mrt_entrance_count.sum()),'mrt_zero_villages':int(f.observed_mrt_entrance_count.eq(0).sum()),'mrt500_area_weighted_ratio':float(f.mrt_entrance_buffer_500m_area_m2.sum()/(a*1e6))})
u.show('12行政區品質，密度與比例採明示分母加權',pd.DataFrame(quality),block=12)


**七核心及MRT／號誌敏感度完整統計**

,feature,valid_N,missing_N,zero_N,min,q25,median,q75,max,mean,std_ddof1
0,osm_road_density,456,0,0,2.469895,19.585107,27.226714,33.618423,48.868213,25.908186,9.911899
1,osm_major_road_ratio,456,0,238,0.000000,0.000000,0.000000,0.135074,0.488236,0.076480,0.106696
2,observed_signal_location_density,456,0,27,0.000000,8.674376,19.606408,33.635733,116.584077,22.441216,17.400425
3,crosswalk_area_ratio,456,0,0,0.000019,0.002621,0.006484,0.009135,0.019291,0.006334,0.004110
4,median_annotated_road_width,453,3,0,5.450000,8.000000,8.000000,10.000000,37.000000,9.282748,3.468611
5,mrt_entrance_density,456,0,318,0.000000,0.000000,0.000000,2.646806,32.821462,2.631295,5.343690
6,mrt_entrance_buffer_500m_ratio,456,0,108,0.000000,0.021182,0.546809,0.957637,1.000000,0.509287,0.409065
7,mrt_entrance_buffer_300m_ratio,456,0,162,0.000000,0.000000,0.108513,0.455646,1.000000,0.249161,0.298506
8,mrt_entrance_buffer_800m_ratio,456,0,62,0.000000,0.388752,1.000000,1.000000,1.000000,0.717817,0.395824
9,signal_geometry_only_location_density,456,0,23,0.000000,9.301856,21.360162,35.032442,116.584077,23.615300,17.769757


**分母與標註樣本數分布**

,feature,valid_N,missing_N,zero_N,min,q25,median,q75,max,mean,std_ddof1
0,fixed_village_area_km2,456,0,0,0.031744,0.134541,0.207946,0.424798,16.324434,0.588926,1.351749
1,osm_road_length_km,456,0,0,0.635487,4.086434,5.835382,9.153730,58.069476,8.127639,7.464725
2,observed_signal_record_count,456,0,27,0.000000,2.000000,5.000000,8.000000,46.000000,5.649123,4.826179
3,observed_unique_signal_location_count,456,0,27,0.000000,2.000000,5.000000,8.000000,46.000000,5.631579,4.784615
4,annotated_road_width_sample_count,456,0,3,0.000000,34.000000,46.000000,62.250000,190.000000,49.125000,24.713473
5,observed_mrt_entrance_count,456,0,318,0.000000,0.000000,0.000000,1.000000,12.000000,0.614035,1.211729


**12行政區品質，密度與比例採明示分母加權**

,district,villages,road_km_per_km2,major_ratio_length_weighted,signal_A_locations,signal_B_locations,district_conflict_records,width_samples,width_NA_villages,mrt_entrances,mrt_zero_villages,mrt500_area_weighted_ratio
0,中山區,42,24.866529,0.146997,260,267,7,2416,0,34,24,0.491194
1,中正區,31,29.580536,0.157649,190,216,26,1830,0,51,14,0.762085
2,信義區,41,19.083768,0.073922,156,179,25,1720,0,24,29,0.499925
3,內湖區,39,12.359207,0.108129,304,304,0,2355,0,15,27,0.186619
4,北投區,42,8.462733,0.072493,239,239,0,2204,2,20,31,0.118348
5,南港區,20,11.080300,0.142901,189,194,5,883,0,18,15,0.168682
6,士林區,51,8.744761,0.057097,266,271,5,2370,1,7,47,0.042373
7,大同區,25,31.136420,0.169798,146,150,4,1198,0,21,17,0.586144
8,大安區,53,29.353804,0.113941,216,246,30,2682,0,52,26,0.676858
9,文山區,43,12.620411,0.111975,236,237,1,1951,0,13,33,0.168099


### 品質、Readiness與零／NA／Blocked

In [6]:
u.show('逐特徵Readiness Matrix；不代表DA／PM已驗收',pd.DataFrame(contract['feature_readiness']))
u.show('Blocked：沒有假造數值欄',pd.DataFrame(contract['blocked_features']))
u.show('實際道路fclass定義／研究納排，major沿Unit1.7六類',pd.DataFrame(contract['road_classification']))
u.show('城市union與逐里和／重疊上界核對，非互斥假設',pd.DataFrame(contract['spatial_reconciliation']))
u.show('無路寬樣本：不是零寬度',frame.loc[frame.annotated_road_width_no_valid_samples,['village_id','district_name','standard_village_name','annotated_road_width_sample_count','median_annotated_road_width','annotated_road_width_q1','annotated_road_width_q3']])
u.show('糖廍及士林公館完整保留',frame.loc[frame.village_id.isin(['6300700015','6301100044']),['village_id','standard_village_name','verification_status']+u.CORE])
print('微量覆蓋比例超界里數／最大超額m²',int(frame.coverage_ratio_roundoff_flag.sum()),float(frame.coverage_ratio_max_excess_m2.max()))
print('來源完整普查認證旗標',frame.source_complete_census_certified.unique().tolist())


**逐特徵Readiness Matrix；不代表DA／PM已驗收**

,feature,source,source_sha256,observation_unit,numerator,denominator,raw_crs,working_crs,crs_evidence,time_reference,spatial_completeness,readiness_status,recommended_analytical_use,key_limitations,human_acceptance,valid_N,missing_N,zero_N,min,q25,median,q75,max,mean,std_ddof1,value_state_counts,spatial_processing_valid_values,formal_analysis_selected
0,osm_road_density,data/raw/taiwan-250101-free.shp.zip,ad19e79129efc0a447db5fe5d684e732199a900251f04f2aacef821cfa9ff1ab,specified motor-candidate line representation,same-class-independent all included line union km excluding shared boundary,fixed_village_area_km2,EPSG:4326,EPSG:3826,A metadata,包內README明示2025-01-01T21:21:00Z快照；非2024／2025逐年獨立路網,source coverage not certified as a full census; frozen original villages preserved,Ready with explicit assumptions,priority external comparison candidate,"not centerline / traffic / VKT; fixed OSM snapshot, 2D union, class and boundary assumptions",DE / DA / PM pending,456,0,0,2.469895,19.585107,27.226714,33.618423,48.868213,25.908186,9.911899,{'observed_positive_with_source_limits': 456},456,False
1,osm_major_road_ratio,data/raw/taiwan-250101-free.shp.zip,ad19e79129efc0a447db5fe5d684e732199a900251f04f2aacef821cfa9ff1ab,six-class narrow research major line representation,major line union km in same primary boundary policy,osm_road_length_km,EPSG:4326,EPSG:3826,A metadata,包內README明示2025-01-01T21:21:00Z快照；非2024／2025逐年獨立路網,source coverage not certified as a full census; frozen original villages preserved,Ready with explicit assumptions,priority external comparison candidate,"Unit1.7 six classes, not official road hierarchy or traffic share; no road denominator => NA",DE / DA / PM pending,456,0,238,0.000000,0.000000,0.000000,0.135074,0.488236,0.076480,0.106696,"{'observed_zero_not_confirmed_absent': 238, 'observed_positive_with_source_limits': 218}",456,False
2,observed_signal_location_density,data/raw/號誌路口經緯度一覽20230706.csv,4de42a369f6b6ba38df29ff2d357f8ad5aec7a237b8709edc431d77c3cfb29a7,unique exact coordinate locations in conservative A policy,"readable non-test, unique interior, all basic member districts recognized and matching locations",fixed_village_area_km2,EPSG:4326 working interpretation,EPSG:3826,B working convention,檔名20230706非每筆觀測／設置時間；實際快照時點仍未認證,source coverage not certified as a full census; frozen original villages preserved,Partially ready,conditional priority; DA review of A/B and LCS sensitivity,"not physical intersections; uncertain LCS meanings/date/completeness, duplicate and source district conflicts retained",DE / DA / PM pending,456,0,27,0.000000,8.674376,19.606408,33.635733,116.584077,22.441216,17.400425,"{'observed_positive_with_source_limits': 429, 'observed_zero_not_confirmed_absent': 23, 'locations_excluded_by_primary_quality_policy': 4}",456,False
3,crosswalk_area_ratio,data/raw/(交工處)行人穿越線圖資_202407121519.zip,a7f4241ea1ecccfb09050fb38f4d8537a7f5ec1d4134eb7b8929217601ba0493,six-layer crossing-marking polygon union,observed marking union area_m2,fixed_village_area_m2,EPSG:3826,EPSG:3826,A metadata,2024-07-12封裝標記，非全年有效或2025新增完整紀錄；RDDATE需排除1899占位,source coverage not certified as a full census; frozen original villages preserved,Partially ready,supplementary observed markings comparison,components not full facilities; source coverage and historical stock unknown,DE / DA / PM pending,456,0,0,0.000019,0.002621,0.006484,0.009135,0.019291,0.006334,0.004110,{'observed_positive_with_source_limits': 456},456,False
4,median_annotated_road_width,data/raw/路寬檔_提供.zip,a92d7b7225509489181606158d1d9cbe3b1a4c83b120ee12d577460109852d70,"unique exact geometry + width annotation, assigned by midpoint",median of unique-midpoint valid positive annotation widths,median operator over sample count,EPSG:3826,EPSG:3826,A metadata,ZIP內資料修改2023-12-28，XML製作2021；非2024／2025完整年度實測,source coverage not certified as a full census; frozen original villages preserved,Partially ready,supplementary planned annotation sample description,not measured road wid

**Blocked：沒有假造數值欄**

,feature,readiness_status,key_limitations,formal_numeric_column_created,recommended_analytical_use
0,intersection_density,Blocked,缺完整節點／轉向／立體拓樸，平面交叉不當實體路口,False,do not use
1,complete_topological_intersection_count,Blocked,免費線圖層缺完整節點與立體道路拓樸,False,do not use
2,complete_crosswalk_count,Blocked,一Polygon不是完整設施,False,do not use
3,crosswalk_length_density,Blocked,標線面構件不提供可靠穿越長度,False,do not use
4,length_weighted_road_width,Blocked,尺寸線不是縱向道路權重,False,do not use
5,narrow_road_length_ratio,Blocked,無可靠縱向分母與窄寬門檻,False,do not use
6,wide_road_length_ratio,Blocked,無可靠縱向分母與窄寬門檻,False,do not use
7,marked_sidewalk_density,Blocked,混合標線未驗證人行道子集,False,do not use
8,complete_signalized_intersection_count,Blocked,位置紀錄未認證實體路口單位,False,do not use
9,marked_sidewalk_area_ratio,Blocked,混合標線未驗證人行道子集，不能計算正式面積或覆蓋率,False,do not use


**實際道路fclass定義／研究納排，major沿Unit1.7六類**

,fclass,source_definition,research_group,included,major,inclusion_reason,bbox_source_records,regional_records,source_documentation
0,bridleway,騎馬路徑,walking,False,False,步行／自行車／產業道路或未知類別，依Unit1.7不納主要機動候選,2,2,https://download.geofabrik.de/osm-data-in-gis-formats-free.pdf
1,busway,公車專用道路,service,True,False,Unit1.7機動車候選研究規格,34,34,https://download.geofabrik.de/osm-data-in-gis-formats-free.pdf
2,cycleway,自行車道,cycling,False,False,步行／自行車／產業道路或未知類別，依Unit1.7不納主要機動候選,1510,1263,https://download.geofabrik.de/osm-data-in-gis-formats-free.pdf
3,footway,步行路徑,walking,False,False,步行／自行車／產業道路或未知類別，依Unit1.7不納主要機動候選,24965,21975,https://download.geofabrik.de/osm-data-in-gis-formats-free.pdf
4,living_street,行人優先街道,local,True,False,Unit1.7機動車候選研究規格,298,146,https://download.geofabrik.de/osm-data-in-gis-formats-free.pdf
5,motorway,高速道路,major,True,True,Unit1.7機動車候選研究規格,412,260,https://download.geofabrik.de/osm-data-in-gis-formats-free.pdf
6,motorway_link,高速道路連接匝道,major,True,True,Unit1.7機動車候選研究規格,462,310,https://download.geofabrik.de/osm-data-in-gis-formats-free.pdf
7,path,未指定用途路徑,walking,False,False,步行／自行車／產業道路或未知類別，依Unit1.7不納主要機動候選,5151,4116,https://download.geofabrik.de/osm-data-in-gis-formats-free.pdf
8,pedestrian,行人專用街道,walking,False,False,步行／自行車／產業道路或未知類別，依Unit1.7不納主要機動候選,739,602,https://download.geofabrik.de/osm-data-in-gis-formats-free.pdf
9,primary,主要道路,major,True,True,Unit1.7機動車候選研究規格,1271,879,https://download.geofabrik.de/osm-data-in-gis-formats-free.pdf


... (4 outputs left out)

,village_id,district_name,standard_village_name,annotated_road_width_sample_count,median_annotated_road_width,annotated_road_width_q1,annotated_road_width_q3
410,6301100048,士林區,平等里,0,<NA>,<NA>,<NA>
453,6301200040,北投區,湖山里,0,<NA>,<NA>,<NA>
455,6301200042,北投區,湖田里,0,<NA>,<NA>,<NA>


**糖廍及士林公館完整保留**

,village_id,standard_village_name,verification_status,osm_road_density,osm_major_road_ratio,observed_signal_location_density,crosswalk_area_ratio,median_annotated_road_width,mrt_entrance_buffer_500m_ratio,mrt_entrance_density
239,6300700015,糖廍里,provisional_match,32.733597,0.116269,13.203685,0.008144,6.0,0.655057,0.0
406,6301100044,公館里,verified_exact_match,8.60042,0.0,0.0,0.000189,10.0,0.0,0.0


微量覆蓋比例超界里數／最大超額m² 93 0.00011237026774324477
來源完整普查認證旗標 [False]


### 使用方式及技術交付

In [7]:
handoff=u.handoff_check(PROJECT_ROOT,frame,HANDOFF_ONLY_INPUTS)
u.show('七份A／B1／B2／C只讀鍵與必要面積，未寫整合表',pd.DataFrame(handoff))
print('同一B3快照接不同year／事故spec不構成年度獨立環境觀測；2026MRT不得當事前模型輸入。')

**七份A／B1／B2／C只讀鍵與必要面積，未寫整合表**

,dataset,rows_before,rows_after,unmatched,distinct_villages,merge,area_fields_verified,common_village_base_hash_verified,area_scope,B3_partial_or_missing_rows_retained,source_feature_values_used_for_B3,snapshot_repeated_not_annual_observation
0,taipei_dataset_a_annual,3648,3648,0,456,many_to_one,[],True,composition ratios have no area denominator; same fixed village-base provenance verified,24,False,True
1,taipei_dataset_a_pooled,1824,1824,0,456,many_to_one,[],True,composition ratios have no area denominator; same fixed village-base provenance verified,12,False,True
2,taipei_dataset_b1_annual,912,912,0,456,many_to_one,[fixed_village_area_km2],True,direct fields checked,6,False,True
3,taipei_dataset_b1_pooled,456,456,0,456,many_to_one,[fixed_village_area_km2],True,direct fields checked,3,False,True
4,taipei_dataset_c_annual,3648,3648,0,456,many_to_one,[fixed_village_area_km2],True,direct fields checked,24,False,True
5,taipei_dataset_c_pooled,1824,1824,0,456,many_to_one,[fixed_village_area_km2],True,direct fields checked,12,False,True
6,taipei_dataset_b2,456,456,0,456,many_to_one,"[fixed_village_area_m2, fixed_village_area_km2]",True,direct fields checked,3,False,True


同一B3快照接不同year／事故spec不構成年度獨立環境觀測；2026MRT不得當事前模型輸入。


### Assertions

獨立saved-file檢查，不重跑GIS；完成停止Unit2.6。

In [8]:
u.validate_dataset(frame,base);u.verify(PROJECT_ROOT,B3_INPUTS);u.verify(PROJECT_ROOT,HANDOFF_ONLY_INPUTS)
for rel,h in EXPECTED_OUTPUTS.items():assert sha(PROJECT_ROOT/rel)==h
again=u.read_csv(PROJECT_ROOT/'data/processed/taipei_dataset_b3.csv',contract)
pd.testing.assert_frame_equal(again,frame,check_exact=True)
assert set(contract['core_candidate_features'])==set(u.CORE)
for row in contract['blocked_features']:assert row['feature'] not in frame and row['formal_numeric_column_created'] is False
result={'status':'passed','shape':list(frame.shape),'independent_saved_file_read':True,'csv_sha256':sha(PROJECT_ROOT/'data/processed/taipei_dataset_b3.csv'),'schema_sha256':sha(PROJECT_ROOT/'data/processed/taipei_dataset_b3_schema.json'),'producer_sha256':sha(PROJECT_ROOT/'data/code/prepare_taipei_dataset_b3.ipynb'),'human_acceptance':'DE / DA / PM pending'}
temp=PROJECT_ROOT/'data/temp/unit26_processing';temp.mkdir(parents=True,exist_ok=True)
(temp/'documentation_summary.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print(json.dumps(result,ensure_ascii=False,indent=2))


{
  "status": "passed",
  "shape": [
    456,
    113
  ],
  "independent_saved_file_read": true,
  "csv_sha256": "36d6f42a86a54ba39552c611332c5185c89a3740e14f5e9e0268158ec8d9debe",
  "schema_sha256": "440f6020e336990071a3492eba2f42ea36229f11948a8ea14f59427c9e25b8bf",
  "producer_sha256": "c10ccbd944f96710af4cbb25b7bb980bf80ef50e109ecbae8a40a863880c4f06",
  "human_acceptance": "DE / DA / PM pending"
}
